In [150]:
import optuna
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error,confusion_matrix,classification_report,accuracy_score
from xgboost import XGBRegressor,XGBClassifier
import json
import optuna_classfier

In [ ]:
data = pd.read_csv(r'C:\Users\udot\OneDrive\Desktop\New folder\AI-Powered-Student-Achievement-Intelligence-Platform\dataset\student-por.csv')
data.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [ ]:
df = data.drop(columns=["G1", "G2"])

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 649 entries, 0 to 648
Data columns (total 31 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   school      649 non-null    str  
 1   sex         649 non-null    str  
 2   age         649 non-null    int64
 3   address     649 non-null    str  
 4   famsize     649 non-null    str  
 5   Pstatus     649 non-null    str  
 6   Medu        649 non-null    int64
 7   Fedu        649 non-null    int64
 8   Mjob        649 non-null    str  
 9   Fjob        649 non-null    str  
 10  reason      649 non-null    str  
 11  guardian    649 non-null    str  
 12  traveltime  649 non-null    int64
 13  studytime   649 non-null    int64
 14  failures    649 non-null    int64
 15  schoolsup   649 non-null    str  
 16  famsup      649 non-null    str  
 17  paid        649 non-null    str  
 18  activities  649 non-null    str  
 19  nursery     649 non-null    str  
 20  higher      649 non-null    str  
 21  inte

In [ ]:
df.isnull().sum()

school        0
sex           0
age           0
address       0
famsize       0
Pstatus       0
Medu          0
Fedu          0
Mjob          0
Fjob          0
reason        0
guardian      0
traveltime    0
studytime     0
failures      0
schoolsup     0
famsup        0
paid          0
activities    0
nursery       0
higher        0
internet      0
romantic      0
famrel        0
freetime      0
goout         0
Dalc          0
Walc          0
health        0
absences      0
G3            0
dtype: int64

In [ ]:
for col in df.columns:
    print(f"\n{col}")
    print("Unique values:", df[col].nunique())
    print(df[col].unique())


school
Unique values: 2
<StringArray>
['GP', 'MS']
Length: 2, dtype: str

sex
Unique values: 2
<StringArray>
['F', 'M']
Length: 2, dtype: str

age
Unique values: 8
[18 17 15 16 19 22 20 21]

address
Unique values: 2
<StringArray>
['U', 'R']
Length: 2, dtype: str

famsize
Unique values: 2
<StringArray>
['GT3', 'LE3']
Length: 2, dtype: str

Pstatus
Unique values: 2
<StringArray>
['A', 'T']
Length: 2, dtype: str

Medu
Unique values: 5
[4 1 3 2 0]

Fedu
Unique values: 5
[4 1 2 3 0]

Mjob
Unique values: 5
<StringArray>
['at_home', 'health', 'other', 'services', 'teacher']
Length: 5, dtype: str

Fjob
Unique values: 5
<StringArray>
['teacher', 'other', 'services', 'health', 'at_home']
Length: 5, dtype: str

reason
Unique values: 4
<StringArray>
['course', 'other', 'home', 'reputation']
Length: 4, dtype: str

guardian
Unique values: 3
<StringArray>
['mother', 'father', 'other']
Length: 3, dtype: str

traveltime
Unique values: 4
[2 1 3 4]

studytime
Unique values: 4
[2 3 1 4]

failures
Unique 

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
correlation = df[numeric_cols].corr()["G3"].sort_values(ascending=False)
correlation

G3            1.000000
studytime     0.249789
Medu          0.240151
Fedu          0.211800
famrel        0.063361
goout        -0.087641
absences     -0.091379
health       -0.098851
age          -0.106505
freetime     -0.122705
traveltime   -0.127173
Walc         -0.176619
Dalc         -0.204719
failures     -0.393316
Name: G3, dtype: float64

In [ ]:
numeric_features = df.select_dtypes(include=np.number).columns.tolist()
categorical_features = df.select_dtypes(include=["object"]).columns.tolist()
print("Numerical Features:")
print(numeric_features)
print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G3']

Categorical Features:
['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']


C:\Users\udot\AppData\Local\Temp\ipykernel_17020\3111186415.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = df.select_dtypes(include=["object"]).columns.tolist()


In [ ]:
print(df.shape)
print("G3" in df.columns)

(649, 31)
True


In [ ]:
X = df.drop(columns=["G3"])
y = df["G3"]
X.head()
y.head()

0    11
1    11
2    12
3    14
4    13
Name: G3, dtype: int64

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "str"]).columns.tolist()
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ]
)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
print("Original X_train shape:", X_train.shape)
print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape:", X_test_processed.shape)

Original X_train shape: (519, 30)
Processed X_train shape: (519, 56)
Processed X_test shape: (130, 56)


In [ ]:

def model(X_train_processed, Y_train, X_test_processed, Y_test):
    def objective(trial):
        params = {
            "objective": "reg:squarederror",
            "n_estimators": trial.suggest_int("n_estimators", 100, 800),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "gamma": trial.suggest_float("gamma", 0, 5),
            "reg_alpha": trial.suggest_float("reg_alpha", 0, 2),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 5),
        }

        regressor = XGBRegressor(**params, random_state=42)
        regressor.fit(X_train_processed, Y_train)
        predictions = regressor.predict(X_test_processed)
        rmse = mean_squared_error(Y_test, predictions) ** 0.5
        return rmse

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=50)

    best_params = study.best_params
    best_model = XGBRegressor(**best_params, random_state=42)
    best_model.fit(X_train_processed, Y_train)
    y_pred = best_model.predict(X_test_processed)
    rmse = mean_squared_error(Y_test, y_pred) ** 0.5

    print("Best RMSE:", study.best_value)
    print("Best Parameters:")
    print(best_params)

    return best_model, y_pred, rmse, best_params


In [ ]:
best_model, y_pred, rmse, best_params = model(
    X_train_processed,
    Y_train,
    X_test_processed,
    Y_test
)

[I 2026-09-10 16:15:45,132] A new study created in memory with name: no-name-f11327cb-4f6d-4087-8235-595b3b1b06be
[I 2026-09-10 16:15:45,547] Trial 0 finished with value: 2.7733203272178315 and parameters: {'n_estimators': 121, 'learning_rate': 0.018710453574113408, 'max_depth': 3, 'min_child_weight': 4, 'subsample': 0.8161210538895481, 'colsample_bytree': 0.9654010207422365, 'gamma': 3.909237554170415, 'reg_alpha': 1.4829140758376589, 'reg_lambda': 0.48631116936919505}. Best is trial 0 with value: 2.7733203272178315.
[I 2026-09-10 16:15:47,080] Trial 1 finished with value: 2.7769836964992356 and parameters: {'n_estimators': 431, 'learning_rate': 0.09942877846138176, 'max_depth': 3, 'min_child_weight': 5, 'subsample': 0.9561969683399906, 'colsample_bytree': 0.9056446863813675, 'gamma': 3.8506421353122584, 'reg_alpha': 1.787150554692925, 'reg_lambda': 4.295383314330049}. Best is trial 0 with value: 2.7733203272178315.
[I 2026-09-10 16:15:47,486] Trial 2 finished with value: 2.7401158693

Best RMSE: 2.6312078912043115
Best Parameters:
{'n_estimators': 593, 'learning_rate': 0.08207529631283852, 'max_depth': 9, 'min_child_weight': 7, 'subsample': 0.8869019086855202, 'colsample_bytree': 0.8432588990427722, 'gamma': 3.125784483402925, 'reg_alpha': 0.524273638026089, 'reg_lambda': 0.6243396325621215}


In [ ]:
with open("best_params.json", "w") as f:
    json.dump(best_params, f, indent=4)

print("Best parameters saved successfully!")

Best parameters saved successfully!


In [ ]:
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
print("MSE:", mse)
print("RMSE:", rmse)
print("MAE:", mae)
print("R2 score:", r2)

MSE: 6.92325496673584
RMSE: 2.6312078912043115
MAE: 1.9683181047439575
R2 score: 0.2900475859642029


In [ ]:
results = pd.DataFrame({
    "Actual G3": Y_test.values,
    "Predicted G3": y_pred
})
results.head()

,Actual G3,Predicted G3
0,19,12.353027
1,12,12.277081
2,18,14.102877
3,11,11.599263
4,11,11.766225


In [ ]:
def risk_category(score):
    if score < 10:
        return 0
    elif score < 14:
        return 1
    else:
        return 2


Y_train_risk = Y_train.apply(risk_category)
Y_test_risk = Y_test.apply(risk_category)

print(Y_train_risk.value_counts())
print(Y_test_risk.value_counts())

G3
1    284
2    150
0     85
Name: count, dtype: int64
G3
1    71
2    44
0    15
Name: count, dtype: int64


In [ ]:
classifier = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

classifier.fit(X_train_processed, Y_train_risk)

Y_pred_risk = classifier.predict(X_test_processed)

accuracy = accuracy_score(Y_test_risk, Y_pred_risk)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(
    Y_test_risk,
    Y_pred_risk,
    target_names=["High Risk", "Medium Risk", "Low Risk"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(Y_test_risk, Y_pred_risk))

Accuracy: 0.6384615384615384

Classification Report:
              precision    recall  f1-score   support

   High Risk       0.40      0.27      0.32        15
 Medium Risk       0.69      0.72      0.70        71
    Low Risk       0.61      0.64      0.62        44

    accuracy                           0.64       130
   macro avg       0.57      0.54      0.55       130
weighted avg       0.63      0.64      0.63       130


Confusion Matrix:
[[ 4  8  3]
 [ 5 51 15]
 [ 1 15 28]]


In [ ]:
from sklearn.metrics import f1_score

def objective(trial):
    params = {
        "objective": "multi:softprob",
        "num_class": 3,
        "eval_metric": "mlogloss",
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42,
    }

    model = XGBClassifier(**params)
    model.fit(X_train_processed, Y_train_risk)
    predictions = model.predict(X_test_processed)

    return f1_score(Y_test_risk, predictions, average="macro")


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

best_classifier = XGBClassifier(
    **study.best_params,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42
)

best_classifier.fit(X_train_processed, Y_train_risk)
Y_pred_risk = best_classifier.predict(X_test_processed)
macro_f1 = f1_score(Y_test_risk, Y_pred_risk, average="macro")

print("Optimized Macro F1:", macro_f1)

print("Optimized Macro F1:", macro_f1)

AttributeError: module 'optuna_classfier' has no attribute 'classifier'